In [ ]:
min_valloss = 0.047911887775863644

In [ ]:
# RUN THIS CELL FIRST
import os
import math
from collections import OrderedDict, Counter

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import v2

import numpy as np
from numpy import allclose, isclose

from collections.abc import Callable
from sklearn.model_selection import train_test_split

TRAIN_PATH = "data/train"
VAL_PATH = "data/val"
TEST_PATH = "data/test"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
transformations = transforms.Compose(
    [
        v2.Resize((128,128)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.RandomHorizontalFlip(),
        v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        v2.RandomErasing(p=0.5, scale=(0.1, 0.3)),
        v2.RandomCrop((112,112)),
        v2.RandomAdjustSharpness(2),
        v2.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
    ]
)

val_transformation = v2.Compose(
    [
        v2.Resize((128,128)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.CenterCrop((112, 112)),
        v2.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
    ]
)

train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=transformations)
val_dataset = datasets.ImageFolder(VAL_PATH, transform=val_transformation)
test_dataset = datasets.ImageFolder(TEST_PATH, transform=val_transformation)

class_labels = list(train_dataset.class_to_idx.keys())
indices = list(range(len(train_dataset)))
labels = [y for _, y in train_dataset]
class_counts = Counter(labels)
weights = [1 / class_counts[i] for i in labels]
num_classes = torch.unique(torch.tensor(labels))

In [ ]:
print(class_counts)
print(train_dataset.class_to_idx)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
print(len(weights))


In [ ]:
def get_accuracy(scores: torch.Tensor, labels: torch.Tensor) -> int | float:
    _, predictions = torch.max(scores, 1)
    # Per-class accuracy breakdown
    classes = torch.unique(labels)
    for cls in classes:
        mask = labels == cls
        cls_acc = (predictions[mask] == labels[mask]).float().mean().item()
        print(f"Class {class_labels[cls.item()]}: {cls_acc:.2%}")

    overall = (predictions == labels).float().mean().item()
    print(f"Overall: {overall:.2%}")
    return overall

In [10]:
class TestCNN(nn.Module):
    def __init__(self, classes: int):
        super().__init__()
        self.conv = nn.Sequential(
                        nn.Conv2d(3, 32, (3,3), padding=1, stride=2),
                        nn.BatchNorm2d(32),
                        nn.LeakyReLU(0.01),
                        nn.Dropout(0.5),
                        nn.Conv2d(32, 32, (3,3), padding=1, stride=2),
                        nn.BatchNorm2d(32),
                        nn.LeakyReLU(0.01),
                        nn.Dropout(0.5),
                        nn.Conv2d(32, 64, (3,3), padding=1, stride=2),
                        nn.BatchNorm2d(64),
                        nn.LeakyReLU(0.01),
                        nn.Dropout(0.5),
                        nn.Conv2d(64, 128, (3,3), padding=1, stride=2),
                        nn.BatchNorm2d(128),
                        nn.LeakyReLU(0.01),
                        nn.Dropout(0.5),
                        nn.Conv2d(128, 128, (3,3), padding=1, stride=2),
                        nn.LeakyReLU(0.01),

                    )

        self.fc = nn.Sequential(
                    nn.Dropout(0.5),
                    nn.Linear(128, 64),
                    nn.LeakyReLU(0.01),
                    nn.Dropout(0.5),
                    nn.Linear(64, classes),
                )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ YOUR CODE HERE """
        x = self.conv(x)
        """ YOUR CODE END HERE """
        x = self.gap(x) # GAP – do not remove this line
        """ YOUR CODE HERE """
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        """ YOUR CODE END HERE """
        return out

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        out = self.forward(x)
        return torch.softmax(out, dim = 1)

In [ ]:
test_model = TestCNN(14).to(device)
optimiser = torch.optim.SGD(test_model.parameters(), momentum=0.9, lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimiser,
                mode='min',
                factor=0.1,
                patience=10,
)


In [9]:
def train_model(model: nn.Module, dataloader: DataLoader, epochs: int = 20):
    """
    Trains the model for a specified number of epochs/iterations

    Parameters
    ----------
        model: A PyTorch model to be trained
        dataloader : A DataLoader object that provides batches of the training data
        epochs  : Number of epochs, default of 20

    Returns
    -------
        The final model and the loss curve (per epoch)
    """
    global min_valloss, optimiser, scheduler
    losses = []
    total = sum(class_counts.values())
    class_weights = torch.tensor(
        [total / class_counts[i] for i in class_counts.keys()],
        dtype=torch.float32
    ).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    # Set model to training mode.
    # See (https://stackoverflow.com/questions/60018578/what-does-model-eval-do-in-pytorch) if curious.
    model.train()

    # optimiser = torch.optim.AdamW(model.parameters())
    for i in range(epochs):
        epoch_loss = 0.0
        y_train_out = []
        y_label_out = []
        num_batches = 0
        for x_batch, y_batch in dataloader:
            optimiser.zero_grad()
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_output = model(x_batch)
            loss = loss_fn(y_output, y_batch)
            y_train_out += y_output.tolist()
            y_label_out += y_batch.tolist()
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
            num_batches += 1
        epoch_loss /= num_batches
        # get_accuracy(torch.tensor(y_train_out), torch.tensor(y_label_out))
        print(f"Epoch {i+1}/{epochs}, Loss: {epoch_loss:.4f}")
        losses.append(epoch_loss)
        num_batches = 0
        model.eval()
        with torch.no_grad():
            val_loss = 0
            y_total_output = []
            y_total = []
            for i, data in enumerate(test_loader):
                x, y = data
                x, y = x.to(device), y.to(device)
                y_output = model(x)
                y_total_output += y_output.tolist()
                y_total += y.tolist()
                val_loss += loss_fn(y_output, y).item()
                num_batches += 1
            get_accuracy(torch.tensor(y_total_output), torch.tensor(y_total))
        val_loss /= num_batches
        print(f"Val Loss: {val_loss:.4f}")
        scheduler.step(val_loss)
        model.train()
        if val_loss < min_valloss:
            min_valloss = val_loss
            torch.save(test_model.state_dict(), "model_weights_med.pth")



    return model, losses

In [ ]:
test_model, test_losses = train_model(test_model, train_loader, epochs = 100)

In [ ]:
test_model_2, test_losses_2 = train_model(TestCNN(14).to(device), train_loader, epochs = 50)

In [ ]:
test_losses_old = test_losses[0:86]

In [ ]:
min_valloss

In [ ]:
test_model, test_losses2 = train_model(test_model, train_loader, epochs = 50)
test_losses += test_losses2

In [ ]:
test_losses += test_losses2

In [ ]:
test_model_2, test_losses2 = train_model(test_model_2, train_loader, epochs = 20)
test_losses_2 += test_losses2

In [ ]:
test_model_3, test_losses2 = train_model(test_model_3, train_loader, epochs = 1)
test_losses += test_losses2

In [ ]:
torch.save(test_model.state_dict(), "model_weights.pth")

In [ ]:
torch.save(test_model_2.state_dict(), "model_weights2.pth")

In [ ]:
torch.save(test_model_3.state_dict(), "model_weights3.pth")

In [ ]:
test_losses.pop()

In [ ]:
plt.plot(test_losses)

In [ ]:
plt.plot(test_losses)

In [ ]:
plt.plot(test_losses)

In [ ]:
plt.plot(test_losses)

In [12]:
from utils import generate_torch_loader_snippet
saved_model = torch.load("model_weights_med.pth", map_location="cpu")
run_model = TestCNN(14)
run_model.load_state_dict(saved_model)
example_input = torch.tensor([[0.0, 0.0]], dtype=torch.float32)  # minimal example for tracing if needed
snippet = generate_torch_loader_snippet(run_model, prefer="auto", compression="zlib")
print(snippet)


def get_model(device: str = "cpu", dtype: str | None = None):
    """
    Return a TorchScript model loaded from an embedded, base64-encoded compressed blob.
    Self-contained: no need for the original Python class.

    Args:
        device: Where to map the model (e.g., "cpu", "cuda", "cuda:0").
        dtype: Optional dtype to convert parameters/buffers to (e.g., "float32", "float16").
    """
    import base64, io, torch
    import zlib as _z; _decomp = _z.decompress
    _blob_b64 = "..."
    _raw = _decomp(base64.b64decode(_blob_b64))
    buf = io.BytesIO(_raw)
    m = torch.jit.load(buf, map_location=device)
    if dtype is not None:
        dt = getattr(torch, dtype) if isinstance(dtype, str) else dtype
        for p in m.parameters():
            p.data = p.data.to(dt)
        for b in m.buffers():
            b.data = b.data.to(dt)
    m.eval()
    return m



In [ ]:
with torch.no_grad():
    run_model.eval()
    y_total = []
    y_output = []
    for i, data in enumerate(val_loader):
        x, y = data
        x, y = x.to(device), y.to(device)
        pred = run_model.predict_proba(x)
        y_output += pred.tolist()
        y_total += y.tolist()
    acc = get_accuracy(torch.tensor(y_output), torch.tensor(y_total))
    print(f"test accuracy: {acc}")
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    _, preds = torch.max(torch.tensor(y_output), 1)
    cm = confusion_matrix(y_total, preds.numpy(), normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=class_labels)
    fig, ax = plt.subplots(figsize=(12, 10))
    disp.plot(ax=ax, xticks_rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
min_valloss

In [ ]:
from collections import Counter
labels = [label for _, label in train_set]
print(Counter(labels))
